# DriftWatch - Final Model Evaluation

This notebook evaluates the finalized local XGBoost model on the held-out test set.

The model, best iteration, and classification threshold are loaded from the training artifact. The test set is never used for training, early stopping, hyperparameter selection, or threshold selection.

In [6]:
from pathlib import Path

import json

import pandas as pd
import xgboost as xgb
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
 )

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "evaluate":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

DATA_ROOT = PROJECT_ROOT / "driftwatch-bucket" / "data" / "processed"
TEST_PATH = DATA_ROOT / "test" / "test.csv"
MODEL_DIR = PROJECT_ROOT / "artifacts" / "local-training"
MODEL_PATH = MODEL_DIR / "xgboost-model"
METADATA_PATH = MODEL_DIR / "training_metadata.json"

print(f"Project root: {PROJECT_ROOT}")
print(f"Test data: {TEST_PATH}")
print(f"Model: {MODEL_PATH}")

Project root: /home/sagemaker-user/DriftWatch
Test data: /home/sagemaker-user/DriftWatch/driftwatch-bucket/data/processed/test/test.csv
Model: /home/sagemaker-user/DriftWatch/artifacts/local-training/xgboost-model


In [7]:
def load_sagemaker_csv(path):
    dataframe = pd.read_csv(path, header=None)
    labels = dataframe.iloc[:, 0].astype(int)
    features = dataframe.iloc[:, 1:].astype(float)
    return features, labels


X_test, y_test = load_sagemaker_csv(TEST_PATH)
metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))

assert set(y_test.unique()).issubset({0, 1})
assert not X_test.isna().any().any()
assert X_test.shape[1] == metadata["feature_count"]

print(f"Test: {X_test.shape}, churn rate={y_test.mean():.2%}")
print(f"Feature count: {X_test.shape[1]}")
print(f"Selected threshold: {metadata['selected_threshold']:.2f}")
print(f"Best iteration: {metadata['best_iteration']}")

Test: (1057, 30), churn rate=26.58%
Feature count: 30
Selected threshold: 0.60
Best iteration: 39


In [8]:
model = xgb.Booster()
model.load_model(MODEL_PATH)

test_matrix = xgb.DMatrix(X_test)
test_probabilities = model.predict(
    test_matrix,
    iteration_range=(0, metadata["best_iteration"] + 1),
)
selected_threshold = float(metadata["selected_threshold"])
test_predictions = (test_probabilities >= selected_threshold).astype(int)

test_metrics = {
    "pr_auc": average_precision_score(y_test, test_probabilities),
    "f1": f1_score(y_test, test_predictions, zero_division=0),
    "precision": precision_score(y_test, test_predictions, zero_division=0),
    "recall": recall_score(y_test, test_predictions, zero_division=0),
}

print("Final test metrics:")
for metric_name, metric_value in test_metrics.items():
    print(f"  {metric_name}: {metric_value:.4f}")

print("\nConfusion matrix [rows=true, columns=predicted]:")
print(confusion_matrix(y_test, test_predictions))

print("\nClassification report:")
print(classification_report(y_test, test_predictions, digits=4, zero_division=0))

Final test metrics:
  pr_auc: 0.6856
  f1: 0.6383
  precision: 0.5909
  recall: 0.6940

Confusion matrix [rows=true, columns=predicted]:
[[641 135]
 [ 86 195]]

Classification report:
              precision    recall  f1-score   support

           0     0.8817    0.8260    0.8530       776
           1     0.5909    0.6940    0.6383       281

    accuracy                         0.7909      1057
   macro avg     0.7363    0.7600    0.7456      1057
weighted avg     0.8044    0.7909    0.7959      1057



## Final evaluation result

These metrics are computed once against the held-out test set using the model and threshold selected during training.